# Post-May-25 — QRC Anchor-Snapshot Diagnostics

This notebook compares the May-25 final-state TFIM-QRC readout against the first working-prototype upgrade: anchor-snapshot / virtual-node observable collection.

Scope:

- target: `future_rv_20d`;
- compare 6q/PCA-6 and 8q/PCA-8;
- compare final-state features vs anchor-snapshot features;
- use `Z+X+ZZ` observables;
- inspect RMSE, QLIKE, Mincer-Zarnowitz, and reservoir feature diagnostics.

This is not a large hyperparameter sweep. The goal is to decide whether anchor snapshots produce more informative reservoir features before tuning reservoir dynamics.

In [ ]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd

from qpitome_qrc.data.features import FEATURE_COLUMNS
from qpitome_qrc.data.loaders import load_phase2_volatility_data
from qpitome_qrc.data.pca import fit_transform_pca_splits_train_only
from qpitome_qrc.data.splits import chronological_tabular_split
from qpitome_qrc.qrc.tfim_reservoir import (
    TFIMQRCConfig,
    diagnose_reservoir_feature_splits,
    fit_tfim_qrc_regressor,
    make_qrc_sequence_splits,
    summarize_qrc_result,
)

## 1. Load data and build PCA sequence windows

In [ ]:
target = "future_rv_20d"

df = load_phase2_volatility_data()
splits = chronological_tabular_split(df)

pca6 = fit_transform_pca_splits_train_only(
    splits,
    feature_columns=FEATURE_COLUMNS,
    target_columns=[target],
    n_components=6,
    prefix="pca6",
)

pca8 = fit_transform_pca_splits_train_only(
    splits,
    feature_columns=FEATURE_COLUMNS,
    target_columns=[target],
    n_components=8,
    prefix="pca8",
)

sequence_splits_6 = make_qrc_sequence_splits(
    pca6.splits,
    feature_columns=pca6.feature_columns,
    target_column=target,
    lookback_days=40,
)

sequence_splits_8 = make_qrc_sequence_splits(
    pca8.splits,
    feature_columns=pca8.feature_columns,
    target_column=target,
    lookback_days=40,
)

In [ ]:
print("PCA-6 cumulative variance:", pca6.explained_variance["cumulative_explained_variance"].iloc[-1])
print("PCA-8 cumulative variance:", pca8.explained_variance["cumulative_explained_variance"].iloc[-1])

print({name: (X.shape, y.shape) for name, (X, y, dates) in sequence_splits_6.items()})
print({name: (X.shape, y.shape) for name, (X, y, dates) in sequence_splits_8.items()})

## 2. Define four comparison cases

In [ ]:
runs = [
    {
        "run_name": "6q_pca6_zxzz_final",
        "sequence_splits": sequence_splits_6,
        "config": TFIMQRCConfig(
            qubits=6,
            pca_components=6,
            lookback_days=40,
            anchor_count=6,
            anchor_policy="even",
            observable_mode="zxzz",
            trotter_steps_per_anchor=1,
            coupling_scale=0.7,
            transverse_field=0.5,
            evolution_time=0.5,
            ridge_alpha=10.0,
            target_transform="log",
            seed=42,
            collect_anchor_features=False,
        ),
    },
    {
        "run_name": "6q_pca6_zxzz_anchor_snapshots",
        "sequence_splits": sequence_splits_6,
        "config": TFIMQRCConfig(
            qubits=6,
            pca_components=6,
            lookback_days=40,
            anchor_count=6,
            anchor_policy="even",
            observable_mode="zxzz",
            trotter_steps_per_anchor=1,
            coupling_scale=0.7,
            transverse_field=0.5,
            evolution_time=0.5,
            ridge_alpha=10.0,
            target_transform="log",
            seed=42,
            collect_anchor_features=True,
        ),
    },
    {
        "run_name": "8q_pca8_zxzz_final",
        "sequence_splits": sequence_splits_8,
        "config": TFIMQRCConfig(
            qubits=8,
            pca_components=8,
            lookback_days=40,
            anchor_count=8,
            anchor_policy="even",
            observable_mode="zxzz",
            trotter_steps_per_anchor=1,
            coupling_scale=0.7,
            transverse_field=0.5,
            evolution_time=0.5,
            ridge_alpha=10.0,
            target_transform="log",
            seed=42,
            collect_anchor_features=False,
        ),
    },
    {
        "run_name": "8q_pca8_zxzz_anchor_snapshots",
        "sequence_splits": sequence_splits_8,
        "config": TFIMQRCConfig(
            qubits=8,
            pca_components=8,
            lookback_days=40,
            anchor_count=8,
            anchor_policy="even",
            observable_mode="zxzz",
            trotter_steps_per_anchor=1,
            coupling_scale=0.7,
            transverse_field=0.5,
            evolution_time=0.5,
            ridge_alpha=10.0,
            target_transform="log",
            seed=42,
            collect_anchor_features=True,
        ),
    },
]

## 3. Run QRC comparisons

In [ ]:
result_rows = []
diagnostic_rows = []
results = {}

for spec in runs:
    run_name = spec["run_name"]
    config = spec["config"]
    sequence_splits = spec["sequence_splits"]

    print(f"Running {run_name}")
    result = fit_tfim_qrc_regressor(
        sequence_splits,
        config=config,
        target=target,
        verbose=True,
    )
    results[run_name] = result

    row = summarize_qrc_result(result)
    row["run_name"] = run_name
    result_rows.append(row)

    _, y_train, _ = sequence_splits["train"]
    _, y_val, _ = sequence_splits["val"]
    _, y_test, _ = sequence_splits["test"]

    diag = diagnose_reservoir_feature_splits(
        result.train_features,
        result.val_features,
        result.test_features,
        y_train,
        y_val,
        y_test,
    )
    diag.insert(0, "run_name", run_name)
    diagnostic_rows.append(diag)

result_table = pd.DataFrame(result_rows)
diagnostics_table = pd.concat(diagnostic_rows, ignore_index=True)

## 4. Metrics table

In [ ]:
metric_cols = [
    "run_name",
    "qubits",
    "pca_components",
    "anchor_count",
    "observable_mode",
    "collect_anchor_features",
    "n_reservoir_features",
    "train_rmse",
    "val_rmse",
    "test_rmse",
    "train_qlike",
    "val_qlike",
    "test_qlike",
    "train_mz_r2",
    "val_mz_r2",
    "test_mz_r2",
]

result_table[metric_cols].sort_values("test_rmse")

## 5. Reservoir feature diagnostics

In [ ]:
diagnostic_cols = [
    "run_name",
    "split",
    "n_samples",
    "n_features",
    "near_constant_features",
    "feature_std_min",
    "feature_std_median",
    "feature_std_max",
    "effective_rank",
    "condition_number",
    "mean_abs_feature_target_corr",
    "max_abs_feature_target_corr",
    "mean_abs_shift_vs_train",
    "max_abs_shift_vs_train",
]

diagnostics_table[diagnostic_cols]

## 6. Save results

In [ ]:
out_dir = Path("results/tables")
out_dir.mkdir(parents=True, exist_ok=True)

result_table.to_csv(out_dir / "phase2_qrc_anchor_snapshot_comparison.csv", index=False)
diagnostics_table.to_csv(out_dir / "phase2_qrc_anchor_snapshot_diagnostics.csv", index=False)

print("Saved comparison and diagnostics to", out_dir)

## 7. Interpretation rule

Anchor snapshots justify further tuning only if they improve at least one of:

```text
test MZ R²
test QLIKE
feature-target correlation
effective rank without severe train/test shift
```

If snapshots do not improve these diagnostics, the next step is not a parameter sweep; it is heterogeneous fixed TFIM dynamics or a residual readout design.